# ESBL / CR *Klebsiella pneumoniae* Transmission Model

Based on: [Identifying the drivers of multidrug-resistant Klebsiella pneumoniae at a European level (PLOS Comp Bio, 2021)](https://doi.org/10.1371/journal.pcbi.1008446)

## Model Structure

**27 compartments per geographic region**, stratified by:
- **Setting**: Community (C) vs. Hospital (H)
- **Strain**: Wild-type (WT), ESBL, Carbapenem-resistant (CR)
- **Treatment arm**: Untreated (U), Cephalosporins (A), Carbapenems (B)

Plus 3 infected compartments (hospital only, one per strain).

**Output groups** (aggregated for plotting):

| Group | Description |
|-------|-------------|
| `S`   | Susceptible (community + hospital) |
| `CW`  | Colonized with wild-type |
| `CE`  | Colonized with ESBL |
| `CC`  | Colonized with carbapenem-resistant |
| `IWT` | Infected — wild-type (hospital) |
| `IET` | Infected — ESBL (hospital) |
| `ICT` | Infected — carbapenem-resistant (hospital) |

## 1. Setup

In [ ]:
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ---- Ensure the project root is on the path ----
# Adjust this if you run the notebook from a different directory.
PROJECT_ROOT = Path("../../..").resolve()  # pandemic-simulator-compartment/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Silence noisy JAX logs
logging.getLogger("jax").setLevel(logging.WARNING)
logging.getLogger("jax._src").setLevel(logging.WARNING)

print(f"Project root: {PROJECT_ROOT}")

## 2. Load & Validate Configuration

In [ ]:
from compartment.helpers import load_config_from_json
from compartment.validation import load_simulation_config

CONFIG_PATH = Path("example-config.json")

raw_config = load_config_from_json(str(CONFIG_PATH))
sim_job = raw_config["data"]["getSimulationJob"]

print(f"Disease type  : {sim_job['Disease']['disease_type']}")
print(f"Run mode      : {sim_job['run_mode']}")
print(f"Date range    : {sim_job['start_date']}  →  {sim_job['end_date']}")
print(f"Admin zones   : {[z['name'] for z in sim_job['case_file']['admin_zones'])}")

In [ ]:
# Validate and build the processed config (Pydantic + post-processing)
cleaned_config = load_simulation_config(raw_config, disease_type="ESBL")

print(f"Compartments  : {cleaned_config.compartment_list}")
print(f"Time steps    : {cleaned_config.time_steps}")
print(f"Admin units   : {[u['name'] for u in cleaned_config.admin_units]}")
print(f"Initial pop shape: {np.array(cleaned_config.initial_population).shape}  (zones × compartments)")

## 3. Instantiate Model & Run Simulation

In [ ]:
from compartment.models.esbl_jax_model.model import ESBLJaxModel
from compartment.simulation_manager import SimulationManager

model = ESBLJaxModel(cleaned_config)

print("Running ODE solver...")
raw_output = SimulationManager(model).run_simulation()
# raw_output shape: (T, C, R)  — timesteps × compartments × regions
T, C, R = raw_output.shape
print(f"Output shape  : {raw_output.shape}  (timesteps={T}, compartments={C}, regions={R})")

## 4. Build a Tidy DataFrame

In [ ]:
from compartment.helpers import get_simulation_step_size

step  = get_simulation_step_size(model.n_timesteps)
t_arr = np.arange(0.0, float(model.n_timesteps), step)

start = pd.Timestamp(model.start_date)
dates = [start + pd.Timedelta(days=float(t)) for t in t_arr]

zone_names = [u["name"] for u in model.admin_units]
comp_list  = model.compartment_list

# Build tidy DataFrame: one row per (date, compartment, zone)
records = []
for r_idx, zone in enumerate(zone_names):
    for c_idx, comp in enumerate(comp_list):
        records.append(
            pd.DataFrame({
                "date":        dates,
                "zone":        zone,
                "compartment": comp,
                "population":  raw_output[:, c_idx, r_idx],
            })
        )

df = pd.concat(records, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
print(df.head())
print(f"\nDataFrame shape: {df.shape}")

## 5. Aggregate into Output Groups

The model's `COMPARTMENT_DELTA_GROUPING` sums the 27 compartments into 7 reporting groups.

In [ ]:
GROUPING = ESBLJaxModel.COMPARTMENT_DELTA_GROUPING

# Invert: compartment → group
comp_to_group = {
    comp: grp
    for grp, comps in GROUPING.items()
    for comp in comps
}

df["group"] = df["compartment"].map(comp_to_group)

df_grouped = (
    df.groupby(["date", "zone", "group"], as_index=False)["population"]
    .sum()
)

print("Groups:", df_grouped["group"].unique())
df_grouped.head()

## 6. Plots

### 6a. All groups — one panel per zone

In [ ]:
GROUP_COLORS = {
    "S":   "#2196F3",
    "CW":  "#4CAF50",
    "CE":  "#FF9800",
    "CC":  "#F44336",
    "IWT": "#9C27B0",
    "IET": "#795548",
    "ICT": "#607D8B",
}

zones = df_grouped["zone"].unique()
n_zones = len(zones)

fig, axes = plt.subplots(n_zones, 1, figsize=(13, 4 * n_zones), sharex=True)
if n_zones == 1:
    axes = [axes]

for ax, zone in zip(axes, zones):
    sub = df_grouped[df_grouped["zone"] == zone]
    for grp, color in GROUP_COLORS.items():
        g = sub[sub["group"] == grp]
        if g.empty:
            continue
        ax.plot(g["date"], g["population"], label=grp, color=color, linewidth=1.5)
    ax.set_title(zone, fontweight="bold")
    ax.set_ylabel("Population")
    ax.legend(loc="upper right", ncol=4, fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date")
fig.suptitle("ESBL Model — All Compartment Groups", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 6b. Colonization prevalence (% of total population)

In [ ]:
# Compute total population per zone & date
total = (
    df_grouped.groupby(["date", "zone"])["population"]
    .sum()
    .reset_index()
    .rename(columns={"population": "total"})
)

df_prev = df_grouped.merge(total, on=["date", "zone"])
df_prev["prevalence"] = 100.0 * df_prev["population"] / df_prev["total"]

colonized_groups = ["CW", "CE", "CC"]

fig, axes = plt.subplots(n_zones, 1, figsize=(13, 4 * n_zones), sharex=True)
if n_zones == 1:
    axes = [axes]

for ax, zone in zip(axes, zones):
    sub = df_prev[(df_prev["zone"] == zone) & (df_prev["group"].isin(colonized_groups))]
    for grp in colonized_groups:
        g = sub[sub["group"] == grp]
        ax.plot(g["date"], g["prevalence"], label=grp, color=GROUP_COLORS[grp], linewidth=2)
    ax.set_title(zone, fontweight="bold")
    ax.set_ylabel("Colonization (% of total)")
    ax.legend(title="Strain", loc="upper right")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date")
fig.suptitle("Colonization Prevalence by Strain", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 6c. Invasive infections (IWT, IET, ICT) — all zones stacked

In [ ]:
infected_groups = ["IWT", "IET", "ICT"]
INFECTED_LABELS = {"IWT": "WT infected", "IET": "ESBL infected", "ICT": "CR infected"}

# Sum across all zones
df_inf = (
    df_grouped[df_grouped["group"].isin(infected_groups)]
    .groupby(["date", "group"])["population"]
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
for grp in infected_groups:
    g = df_inf[df_inf["group"] == grp]
    ax.plot(g["date"], g["population"], label=INFECTED_LABELS[grp],
            color=GROUP_COLORS[grp], linewidth=2)

ax.set_xlabel("Date")
ax.set_ylabel("Infected individuals (all zones)")
ax.set_title("Invasive Infections by Strain — All Zones Combined", fontweight="bold")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 6d. Treatment arm breakdown — community susceptibles

In [ ]:
# SU_C, SA_C, SB_C — susceptible community by treatment arm
arm_comps = {"Untreated (U)": "SU_C", "Cephalosporins (A)": "SA_C", "Carbapenems (B)": "SB_C"}
arm_colors = {"Untreated (U)": "#607D8B", "Cephalosporins (A)": "#FF9800", "Carbapenems (B)": "#F44336"}

fig, axes = plt.subplots(n_zones, 1, figsize=(13, 4 * n_zones), sharex=True)
if n_zones == 1:
    axes = [axes]

for ax, zone in zip(axes, zones):
    sub = df[(df["zone"] == zone) & (df["compartment"].isin(arm_comps.values()))]
    for label, comp in arm_comps.items():
        g = sub[sub["compartment"] == comp]
        ax.plot(g["date"], g["population"], label=label, color=arm_colors[label], linewidth=1.8)
    ax.set_title(zone, fontweight="bold")
    ax.set_ylabel("Population")
    ax.legend(title="Treatment arm", loc="upper right")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date")
fig.suptitle("Susceptible (Community) — by Treatment Arm", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 7. Summary Statistics at End of Simulation

In [ ]:
end_date = df_grouped["date"].max()
df_end = df_grouped[df_grouped["date"] == end_date].copy()

total_by_zone = df_end.groupby("zone")["population"].sum().rename("total")
df_end = df_end.join(total_by_zone, on="zone")
df_end["pct"] = (100 * df_end["population"] / df_end["total"]).round(3)

pivot = df_end.pivot_table(index="group", columns="zone", values="pct")
print(f"Population % by group at {end_date.date()}:\n")
print(pivot.to_string())

## 8. Sensitivity: Vary `beta` and Observe ESBL Colonization

In [ ]:
from copy import deepcopy

beta_values = [0.005, 0.01027, 0.02, 0.04]

fig, ax = plt.subplots(figsize=(13, 5))

for beta in beta_values:
    # Patch config and re-instantiate model
    patched_raw = deepcopy(raw_config)
    patched_raw["data"]["getSimulationJob"]["Disease"]["beta"] = beta
    patched_config = load_simulation_config(patched_raw, disease_type="ESBL")
    m = ESBLJaxModel(patched_config)
    out = SimulationManager(m).run_simulation()  # (T, C, R)

    # Sum CE compartments (indices 6,7,8) across all regions
    ce_indices = [comp_list.index(c) for c in ESBLJaxModel.COMPARTMENT_DELTA_GROUPING["CE"]]
    ce_total = out[:, ce_indices, :].sum(axis=(1, 2))  # sum over comps and regions
    total_pop = out.sum(axis=(1, 2))
    ce_prev   = 100.0 * ce_total / (total_pop + 1e-10)

    ax.plot(dates, ce_prev, label=f"β={beta}")

ax.set_xlabel("Date")
ax.set_ylabel("ESBL colonization (% total population)")
ax.set_title("Sensitivity to Transmission Rate β", fontweight="bold")
ax.legend(title="β", loc="upper left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()